# Chương 2 — Tiền xử lý dữ liệu bảng và đánh giá định lượng

Chương này so sánh imputation, phát hiện ngoại lai, scaling, categorical encoding, feature selection và giảm chiều. Phần mô hình dùng train/test split có stratify; mọi thao tác học tham số được fit trên train để tránh data leakage. Test set không được dùng để chọn phương pháp.

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((p for p in _candidates if (p / "src").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Không tìm thấy thư mục src/. Hãy mở notebook từ thư mục repository gốc.")
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.data_loading import load_adult_dataset
from src.tabular_analysis import split_feature_types

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", context="notebook")
print("Repository:", REPO_ROOT)

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer
from sklearn.impute import SimpleImputer
from scipy.stats import levene, ks_2samp
from src.tabular_analysis import (
    compare_numeric_imputers, detect_outliers, ks_before_after_outlier_removal,
    binary_target, make_preprocessor, encode_variants
)

# Reload raw data to make this notebook independently runnable.
df = load_adult_dataset()
X, numeric_cols, categorical_cols = split_feature_types(df, target="income")
y = binary_target(df["income"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_SEED
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

## 1. Benchmark 7 cấu hình điền khuyết trên 10% MCAR nhân tạo

Mean, Median, Mode, KNN (k=3,5,10) và MICE/IterativeImputer được so sánh trên **các ô số ban đầu đầy đủ**. Chỉ các ô bị che nhân tạo mới được dùng tính RMSE. KNN được chạy sau StandardScaler để khoảng cách không bị biến lớn lấn át. Các cột phân loại trong pipeline mô hình sẽ được điền bằng mode riêng.

\[ RMSE = \sqrt{\frac{1}{m}\sum_{i=1}^{m}(x_i-\hat{x}_i)^2} \]

$$\mathrm{RMSE}=\sqrt{\frac{1}{|M|}\sum_{(i,j)\in M}\left(x_{ij}-\hat x_{ij}\right)^2}$$

$M$ là tập ô bị che nhân tạo. KNN điền bằng trung bình của $k$ láng giềng gần nhất; MICE lặp việc hồi quy từng biến thiếu theo các biến còn lại.

In [ ]:
imputation_results = compare_numeric_imputers(df, numeric_cols, missing_rate=0.10, random_state=RANDOM_SEED)
display(imputation_results.style.format({"RMSE": "{:.4f}"}))
plt.figure(figsize=(9, 4))
sns.barplot(data=imputation_results, x="RMSE", y="method")
plt.title("RMSE on artificially masked numeric cells (10% MCAR)")
plt.tight_layout(); plt.show()
best_imputer = imputation_results.iloc[0]["method"]
print("Best RMSE method on this experiment:", best_imputer)

**Diễn giải:** RMSE thấp hơn nghĩa là dự đoán gần giá trị thật hơn trong thí nghiệm MCAR này. Kết quả không đồng nghĩa phương pháp đó tối ưu cho các cột phân loại hoặc cho mọi cơ chế thiếu. MICE là phương pháp đa biến; KNN nhạy với lựa chọn thang đo và số láng giềng.

## 2. Phát hiện ngoại lai bằng IQR, Z-score, Isolation Forest, LOF và DBSCAN

Các detector khoảng cách/mô hình được fit trên dữ liệu số đã loại hàng thiếu và scale Robust. Vì thuật toán không hoàn toàn đồng nhất, so sánh cả tỷ lệ bị gắn cờ lẫn Jaccard similarity.

\[ J(A,B)=\frac{|A\cap B|}{|A\cup B|} \]

$$\text{IQR: } x<Q_1-1.5\,\mathrm{IQR}\ \text{hoặc}\ x>Q_3+1.5\,\mathrm{IQR},\qquad \text{Z-score: } |z|=\left|\frac{x-\mu}{\sigma}\right|>3$$

$$s(x,n)=2^{-\mathbb E[h(x)]/c(n)},\qquad c(n)=2H(n-1)-\frac{2(n-1)}{n},\qquad J(A,B)=\frac{|A\cap B|}{|A\cup B|}$$

$h(x)$ là độ dài đường đi của $x$ trong cây cô lập, $H$ là số điều hòa, $s$ là điểm bất thường của Isolation Forest; $J$ là chỉ số Jaccard giữa hai tập ngoại lai. LOF so sánh mật độ cục bộ của điểm với mật độ các láng giềng; DBSCAN coi điểm không thuộc cụm nào là nhiễu.

In [ ]:
outlier_data, outlier_masks, outlier_summary, jaccard_matrix = detect_outliers(
    df, numeric_cols, random_state=RANDOM_SEED
)
display(outlier_summary.style.format({"outlier_rate": "{:.2%}"}))
plt.figure(figsize=(12, 6))
sns.heatmap(jaccard_matrix, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
plt.title("Jaccard similarity between outlier detectors")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
# Compare distribution before and after removal using the IQR mask.
ks_results = ks_before_after_outlier_removal(outlier_data, outlier_masks["IQR"], numeric_cols)
display(ks_results.style.format({"KS_statistic": "{:.4f}", "p_value": "{:.3g}"}))

**Giới hạn diễn giải KS:** tập sau lọc là tập con của tập ban đầu nên hai mẫu không độc lập; p-value KS chỉ dùng như chỉ báo mô tả sự thay đổi phân phối, không phải kiểm định độc lập hoàn hảo. Không xóa ngoại lai mặc định: một bản ghi hiếm có thể là giá trị hợp lệ hoặc thông tin quan trọng.

## 3. So sánh Min-Max, Z-score, Robust và Quantile Transform

Bốn nhóm scaler được fit chỉ trên train. Levene được áp dụng trên phân phối của mỗi biến theo hai nhóm nhãn thu nhập trong train sau biến đổi. Scaling là phép biến đổi đơn điệu/toàn cục; nó không đảm bảo biến sẽ có phương sai đồng nhất giữa hai nhóm.

$$x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}},\qquad x'=\frac{x-\mu}{\sigma},\qquad x'=\frac{x-\operatorname{median}(x)}{\mathrm{IQR}},\qquad x'=G^{-1}\bigl(\hat F(x)\bigr)$$

Quantile Transform đưa dữ liệu về phân phối đích $G$ (đều hoặc chuẩn). Kiểm định Levene so sánh phương sai giữa các nhóm; `scipy` mặc định dùng trung vị làm tâm (dạng Brown–Forsythe).

In [ ]:
scalers = {
    "MinMax [0,1]": MinMaxScaler(),
    "Z-score": StandardScaler(),
    "Robust": RobustScaler(),
    "Quantile-uniform": QuantileTransformer(n_quantiles=min(1_000, len(X_train)), output_distribution="uniform", random_state=RANDOM_SEED),
    "Quantile-normal": QuantileTransformer(n_quantiles=min(1_000, len(X_train)), output_distribution="normal", random_state=RANDOM_SEED),
}
scaling_rows = []
scaling_examples = {}
for name, scaler in scalers.items():
    train_numeric = X_train[numeric_cols].copy()
    test_numeric = X_test[numeric_cols].copy()
    imputer = SimpleImputer(strategy="median")
    train_imputed = imputer.fit_transform(train_numeric)
    test_imputed = imputer.transform(test_numeric)
    transformed = scaler.fit_transform(train_imputed)
    scaling_examples[name] = transformed
    for j, col in enumerate(numeric_cols):
        group0 = transformed[np.asarray(y_train == 0), j]
        group1 = transformed[np.asarray(y_train == 1), j]
        stat, p_value = levene(group0, group1, center="median")
        scaling_rows.append({"scaler": name, "feature": col, "Levene_stat": stat, "p_value": p_value})
scaling_results = pd.DataFrame(scaling_rows)
display(scaling_results.head(20).style.format({"Levene_stat": "{:.4f}", "p_value": "{:.3g}"}))

In [ ]:
# Violin plots for a few representative features under each scaler.
features_to_plot = numeric_cols[:3]
fig, axes = plt.subplots(len(features_to_plot), 1, figsize=(11, 4 * len(features_to_plot)))
if len(features_to_plot) == 1: axes = [axes]
for ax, col in zip(axes, features_to_plot):
    plot_rows = []
    for name, arr in scaling_examples.items():
        j = numeric_cols.index(col)
        values = arr[:, j]
        values = values[np.isfinite(values)]
        values = values[:min(5_000, len(values))]
        plot_rows.extend({"scaler": name, "value": v} for v in values)
    plot_df = pd.DataFrame(plot_rows)
    sns.violinplot(data=plot_df, x="scaler", y="value", ax=ax, cut=0)
    ax.set_title(f"{col} — transformed train distribution")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

**Phân tích:** Min-Max giữ biên theo dữ liệu train nhưng nhạy với ngoại lai; Z-score chuẩn hóa mean/std nhưng vẫn nhạy với đuôi dài; Robust dùng median/IQR nên bền vững hơn; Quantile Transform ép phân phối về uniform/normal nhưng làm thay đổi khoảng cách giữa các giá trị. Levene p-value cần diễn giải theo từng biến và từng nhóm lớp, không dùng làm thước đo tổng quát duy nhất.

## 4. Mã hóa biến phân loại: One-Hot, Ordinal, Target, Binary và Frequency

Target Encoding sử dụng `sklearn.preprocessing.TargetEncoder.fit_transform`, có cross-fitting 5-fold trên train; test chỉ được transform bằng encoder đã fit trên train. Binary encoding được cài đặt bằng bit của mã category để xử lý cột high-cardinality (ví dụ `native_country`).

$$\text{Frequency: } \frac{n_c}{N},\qquad \text{Target: } \hat x_c=\lambda_c\bar y_c+(1-\lambda_c)\bar y,\quad \lambda_c=\frac{n_c}{n_c+m}$$

$n_c$ là số dòng của giá trị $c$, $\bar y_c$ là trung bình nhãn của nhóm $c$, $\bar y$ là trung bình toàn bộ, $m$ là tham số làm mịn. One-Hot tạo $K$ cột cho $K$ giá trị; Binary cần $\lceil\log_2 K\rceil$ cột. Target Encoding được cross-fit trên train để tránh rò rỉ nhãn.

In [ ]:
encoding_variants = encode_variants(X_train, X_test, categorical_cols, y_train)
encoding_summary = []
for name, (train_encoded, test_encoded, names) in encoding_variants.items():
    encoding_summary.append({
        "encoding": name,
        "train_rows": train_encoded.shape[0],
        "encoded_features": train_encoded.shape[1],
        "test_rows": test_encoded.shape[0],
    })
encoding_summary = pd.DataFrame(encoding_summary)
display(encoding_summary)

In [ ]:
# Detect high-cardinality categoricals (>20 distinct non-missing values).
cardinality = X_train[categorical_cols].nunique(dropna=True).sort_values(ascending=False)
display(cardinality.rename("unique_categories").to_frame())
high_cardinality_cols = cardinality[cardinality > 20].index.tolist()
print("High-cardinality columns:", high_cardinality_cols)

### VIF — kiểm tra đa cộng tuyến sau mã hóa

VIF: $VIF_j=1/(1-R_j^2)$. Mỗi biến được hồi quy tuyến tính theo các biến còn lại. VIF cao cho thấy có thể giải thích biến đó bằng các biến khác. Với One-Hot, `drop='first'` giúp tránh bẫy biến giả; VIF vẫn có thể cao do quan hệ cấu trúc. Phân tích giới hạn 20 cột đầu để giữ thời gian hợp lý trên máy 12 GB RAM.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings

# VIF của pipeline One-Hot đầy đủ (numeric + categorical).
vif_preprocessor = make_preprocessor(numeric_cols, categorical_cols, numeric_scaler="standard")
X_vif = vif_preprocessor.fit_transform(X_train)
if hasattr(X_vif, "toarray"):
    X_vif = X_vif.toarray()
X_vif = np.asarray(X_vif, dtype=float)
rng = np.random.default_rng(RANDOM_SEED)
row_idx = rng.choice(len(X_vif), size=min(2_000, len(X_vif)), replace=False)
X_vif_small = X_vif[row_idx]
variance = np.var(X_vif_small, axis=0)
valid_cols = np.where(variance > 1e-10)[0][:20]
X_vif_small = X_vif_small[:, valid_cols]
design = np.column_stack([np.ones(len(X_vif_small)), X_vif_small])
vif_rows = []
for idx in range(1, design.shape[1]):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            vif_value = variance_inflation_factor(design, idx)
    except Exception:
        vif_value = np.nan
    vif_rows.append({"encoded_feature_index": int(valid_cols[idx - 1]), "VIF": float(vif_value) if np.isfinite(vif_value) else np.inf})
vif_results = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)
print("VIF của tối đa 20 cột đầu trong pipeline One-Hot (mẫu tối đa 2.000 dòng):")
display(vif_results)

# So sánh VIF của từng kiểu encoding categorical. Mỗi kiểu chỉ fit trên train;
# lấy mẫu hàng và tối đa 20 cột biến thiên để giảm chi phí tính toán.
encoding_vif_rows = []
for encoding_name, (encoded_train, encoded_test, encoded_names) in encoding_variants.items():
    row_sample = rng.choice(encoded_train.shape[0], size=min(2_000, encoded_train.shape[0]), replace=False)
    encoded_sample = encoded_train[row_sample]
    if hasattr(encoded_sample, "toarray"):
        encoded_sample = encoded_sample.toarray()
    encoded_sample = np.asarray(encoded_sample, dtype=float)
    variances = np.var(encoded_sample, axis=0)
    selected_columns = np.where(variances > 1e-10)[0][:20]
    if len(selected_columns) == 0:
        encoding_vif_rows.append({"encoding": encoding_name, "features_checked": 0, "mean_VIF": np.nan, "max_VIF": np.nan})
        continue
    vif_design = np.column_stack([np.ones(len(encoded_sample)), encoded_sample[:, selected_columns]])
    vif_values = []
    for local_idx in range(1, vif_design.shape[1]):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                value = variance_inflation_factor(vif_design, local_idx)
        except Exception:
            value = np.nan
        vif_values.append(float(value) if np.isfinite(value) else np.inf)
    finite_values = [value for value in vif_values if np.isfinite(value)]
    encoding_vif_rows.append({
        "encoding": encoding_name,
        "features_checked": len(selected_columns),
        "mean_VIF_finite": float(np.mean(finite_values)) if finite_values else np.inf,
        "max_VIF": float(np.max(vif_values)) if vif_values else np.nan,
        "infinite_VIF_count": int(np.isinf(vif_values).sum()),
    })
encoding_vif_results = pd.DataFrame(encoding_vif_rows)
display(encoding_vif_results)
print("Lưu ý: VIF được ước lượng trên mẫu và tối đa 20 cột biến thiên đầu tiên của mỗi encoding; đây là chỉ báo rà soát, không thay thế kiểm tra chuyên sâu toàn bộ ma trận.")


## 5. Feature selection và 5-fold CV F1

Mỗi pipeline bao gồm imputation, scaling/mã hóa, bộ chọn đặc trưng và Logistic Regression. `cross_val_score` fit lại toàn bộ pipeline bên trong từng fold, tránh fit encoder hoặc selector trên validation fold.

$$F=\frac{\sum_i n_i(\bar x_i-\bar x)^2/(k-1)}{\sum_i\sum_j(x_{ij}-\bar x_i)^2/(N-k)},\qquad \chi^2=\sum\frac{(O-E)^2}{E},\qquad I(X;Y)=\sum_{x,y}p(x,y)\log\frac{p(x,y)}{p(x)p(y)}$$

$$\mathrm{Gini}=1-\sum_c p_c^2,\qquad F1=\frac{2PR}{P+R}$$

$F$ là thống kê ANOVA, $\chi^2$ là Chi-square, $I$ là thông tin tương hỗ (Mutual Information).

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, chi2, mutual_info_classif, SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
fs_preprocessor = make_preprocessor(numeric_cols, categorical_cols, minmax=True)

def deterministic_mutual_info(features, target):
    return mutual_info_classif(features, target, random_state=RANDOM_SEED)

feature_selection_rows = []
for selector_name, score_func in [
    ("ANOVA F-test", f_classif),
    ("Chi-square", chi2),
    ("Mutual Information", deterministic_mutual_info),
]:
    for k_features in (10, 20, 30, 40):
        pipeline = Pipeline([
            ("preprocess", fs_preprocessor),
            ("select", SelectKBest(score_func=score_func, k=k_features)),
            ("model", LogisticRegression(max_iter=600, solver="liblinear", random_state=RANDOM_SEED)),
        ])
        scores = cross_val_score(pipeline, X_train, y_train, cv=cv5, scoring="f1", n_jobs=1)
        feature_selection_rows.append({
            "method": selector_name, "selected_features": k_features,
            "mean_F1": float(scores.mean()), "std_F1": float(scores.std())
        })

rf_selector = SelectFromModel(
    RandomForestClassifier(n_estimators=120, max_features="sqrt", class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1),
    threshold=-np.inf, max_features=20
)
rf_pipeline = Pipeline([
    ("preprocess", fs_preprocessor), ("select", rf_selector),
    ("model", LogisticRegression(max_iter=600, solver="liblinear", random_state=RANDOM_SEED)),
])
rf_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=cv5, scoring="f1", n_jobs=1)
feature_selection_rows.append({"method": "Random Forest importance", "selected_features": 20, "mean_F1": float(rf_scores.mean()), "std_F1": float(rf_scores.std())})

gb_selector = SelectFromModel(
    GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_SEED),
    threshold=-np.inf, max_features=20
)
gb_pipeline = Pipeline([
    ("preprocess", fs_preprocessor), ("select", gb_selector),
    ("model", LogisticRegression(max_iter=600, solver="liblinear", random_state=RANDOM_SEED)),
])
gb_scores = cross_val_score(gb_pipeline, X_train, y_train, cv=cv5, scoring="f1", n_jobs=1)
feature_selection_rows.append({"method": "Gradient Boosting importance", "selected_features": 20, "mean_F1": float(gb_scores.mean()), "std_F1": float(gb_scores.std())})

feature_selection_results = pd.DataFrame(feature_selection_rows).sort_values(["method", "selected_features"])
display(feature_selection_results.style.format({"mean_F1": "{:.4f}", "std_F1": "{:.4f}"}))
plt.figure(figsize=(9, 5))
sns.lineplot(data=feature_selection_results[feature_selection_results["method"].isin(["ANOVA F-test", "Chi-square", "Mutual Information"])], x="selected_features", y="mean_F1", hue="method", marker="o")
plt.title("5-fold CV F1 versus number of selected features")
plt.ylabel("Mean CV F1"); plt.xlabel("Number of selected features")
plt.tight_layout(); plt.show()


In [ ]:
# Compare Random Forest and Gradient Boosting feature importances on train only.
importance_preprocessor = make_preprocessor(numeric_cols, categorical_cols, numeric_scaler="standard")
X_train_encoded = importance_preprocessor.fit_transform(X_train)
feature_names = importance_preprocessor.get_feature_names_out()
if hasattr(X_train_encoded, "toarray"):
    X_train_encoded = X_train_encoded.toarray()
# A stratified-size cap makes the tree importance study responsive.
from sklearn.model_selection import train_test_split as _split
X_imp, _, y_imp, _ = _split(X_train_encoded, y_train, train_size=min(8_000, len(y_train)), stratify=y_train, random_state=RANDOM_SEED)
rf_model = RandomForestClassifier(n_estimators=180, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1)
gb_model = GradientBoostingClassifier(random_state=RANDOM_SEED)
rf_model.fit(X_imp, y_imp); gb_model.fit(X_imp, y_imp)
importance_table = pd.DataFrame({
    "feature": feature_names,
    "RandomForest": rf_model.feature_importances_,
    "GradientBoosting": gb_model.feature_importances_,
})
display(importance_table.sort_values("RandomForest", ascending=False).head(15))

### Recursive Feature Elimination with Cross-Validation (RFECV)

RFECV lặp lại việc loại bỏ các đặc trưng yếu và dùng cross-validation để chọn số lượng đặc trưng. Để giảm thời gian, bước này chạy trên mẫu train phân tầng tối đa 6.000 dòng; CV bên trong vẫn là 5-fold và scoring F1. Bộ tiền xử lý chỉ fit trên mẫu train dùng cho thí nghiệm này.

In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression

X_rfe_raw, _, y_rfe, _ = train_test_split(
    X_train, y_train, train_size=min(6_000, len(y_train)), stratify=y_train, random_state=RANDOM_SEED
)
rfe_preprocessor = make_preprocessor(numeric_cols, categorical_cols, numeric_scaler="standard")
X_rfe = rfe_preprocessor.fit_transform(X_rfe_raw)
if hasattr(X_rfe, "toarray"):
    X_rfe = X_rfe.toarray()
rfecv = RFECV(
    estimator=LogisticRegression(max_iter=500, solver="liblinear", random_state=RANDOM_SEED),
    step=0.25, min_features_to_select=10,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
    scoring="f1", n_jobs=1,
)
rfecv.fit(X_rfe, y_rfe)
print("RFECV selected feature count:", rfecv.n_features_)
print("RFECV best mean CV F1:", float(np.max(rfecv.cv_results_["mean_test_score"])))
selected_rfe_names = rfe_preprocessor.get_feature_names_out()[rfecv.support_]
print("Selected features:", selected_rfe_names.tolist())

## 6. PCA, t-SNE và UMAP (trực quan hóa)

Adult có trên 30.000 dòng nên trực quan hóa phi tuyến được thực hiện trên mẫu có giới hạn. PCA giữ hướng có phương sai lớn; t-SNE ưu tiên cấu trúc lân cận; UMAP là phép nhúng phi tuyến tùy chọn. Các hình này chỉ dùng khám phá, không phải bằng chứng phân loại hoặc quan hệ nhân quả.

$$\Sigma=\frac{1}{n-1}X_c^\top X_c,\qquad \Sigma v_i=\lambda_i v_i,\qquad Z=X_cV_k$$

$$p_{j|i}=\frac{e^{-\|x_i-x_j\|^2/2\sigma_i^2}}{\sum_{k\neq i}e^{-\|x_i-x_k\|^2/2\sigma_i^2}},\quad q_{ij}=\frac{(1+\|y_i-y_j\|^2)^{-1}}{\sum_{k\neq l}(1+\|y_k-y_l\|^2)^{-1}},\quad \mathrm{KL}(P\|Q)=\sum_{i\neq j}p_{ij}\log\frac{p_{ij}}{q_{ij}}$$

PCA chiếu dữ liệu lên $k$ hướng có phương sai lớn nhất; t-SNE tối thiểu hóa độ phân kỳ KL giữa phân phối láng giềng trong không gian gốc và không gian 2D.

In [ ]:
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.manifold import TSNE

viz_n = min(1_500, X_train_encoded.shape[0])
rng = np.random.default_rng(RANDOM_SEED)
viz_idx = rng.choice(X_train_encoded.shape[0], viz_n, replace=False)
X_viz = X_train_encoded[viz_idx]
y_viz = np.asarray(y_train)[viz_idx]
svd = TruncatedSVD(n_components=min(30, X_viz.shape[1] - 1), random_state=RANDOM_SEED)
X_viz_30 = svd.fit_transform(X_viz)
tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", max_iter=500, random_state=RANDOM_SEED)
X_tsne = tsne.fit_transform(X_viz_30)
plt.figure(figsize=(8, 6))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_viz, cmap="coolwarm", s=10, alpha=0.65)
plt.title("t-SNE of encoded Adult features (train sample)")
plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2"); plt.colorbar(label="income label")
plt.show()

# PCA on the numerical-only scaled matrix; show cumulative variance.
num_imputer = SimpleImputer(strategy="median")
num_scaler = StandardScaler()
X_num_train = num_scaler.fit_transform(num_imputer.fit_transform(X_train[numeric_cols]))
pca = PCA(n_components=min(len(numeric_cols), 6), random_state=RANDOM_SEED)
X_num_pca = pca.fit_transform(X_num_train)
print("PCA cumulative explained variance:", np.cumsum(pca.explained_variance_ratio_))

In [ ]:
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=RANDOM_SEED)
    X_umap = reducer.fit_transform(X_viz_30)
    plt.figure(figsize=(8, 6))
    plt.scatter(X_umap[:, 0], X_umap[:, 1], c=y_viz, cmap="coolwarm", s=10, alpha=0.65)
    plt.title("UMAP of encoded Adult features (train sample)")
    plt.colorbar(label="income label"); plt.show()
except ImportError:
    print("UMAP chưa được cài. Có thể cài bằng: python -m pip install umap-learn")

## 7. [Nâng cao] Class imbalance — resampling chỉ trên train

Tập Adult có thể mất cân bằng giữa hai nhãn. So sánh baseline, SMOTE, ADASYN và Random Under-sampling trên **các biến số** để tránh tạo ra giá trị giả cho biến phân loại đã one-hot. Scaler được fit trên train, resampling chỉ áp dụng trên train; test luôn giữ nguyên phân phối ban đầu.

$$\text{SMOTE: } x_{new}=x_i+\lambda\,(x_{nn}-x_i),\quad \lambda\sim U(0,1)$$

$$P=\frac{TP}{TP+FP},\qquad R=\frac{TP}{TP+FN},\qquad \mathrm{AUC}=\int_0^1 \mathrm{TPR}\,d(\mathrm{FPR})$$

ADASYN nội suy như SMOTE nhưng sinh nhiều mẫu hơn ở vùng khó phân loại; Random Under-sampling bỏ ngẫu nhiên mẫu lớp đa số.

In [ ]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

num_imputer = SimpleImputer(strategy="median")
num_scaler = RobustScaler()
X_num_train = num_scaler.fit_transform(num_imputer.fit_transform(X_train[numeric_cols]))
X_num_test = num_scaler.transform(num_imputer.transform(X_test[numeric_cols]))

samplers = {
    "Baseline": None,
    "SMOTE": SMOTE(random_state=RANDOM_SEED),
    "ADASYN": ADASYN(random_state=RANDOM_SEED),
    "Random Under-sampling": RandomUnderSampler(random_state=RANDOM_SEED),
}
imbalance_rows = []
for name, sampler in samplers.items():
    if sampler is None:
        X_res, y_res = X_num_train, y_train
    else:
        X_res, y_res = sampler.fit_resample(X_num_train, y_train)
    model = LogisticRegression(max_iter=600, class_weight=None, solver="liblinear", random_state=RANDOM_SEED)
    model.fit(X_res, y_res)
    pred = model.predict(X_num_test)
    proba = model.predict_proba(X_num_test)[:, 1]
    imbalance_rows.append({
        "method": name,
        "train_size_after_resampling": len(y_res),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "F1_macro": f1_score(y_test, pred, average="macro", zero_division=0),
        "AUC_ROC": roc_auc_score(y_test, proba),
    })
imbalance_results = pd.DataFrame(imbalance_rows).sort_values("F1_macro", ascending=False)
display(imbalance_results.style.format({"precision": "{:.4f}", "recall": "{:.4f}", "F1_macro": "{:.4f}", "AUC_ROC": "{:.4f}"}))

**Phân tích class imbalance:** không kết luận phương pháp tốt nhất chỉ dựa trên accuracy. So sánh precision, recall, F1-macro và AUC-ROC trên cùng test set nguyên trạng. SMOTE/ADASYN có thể tăng recall nhưng giảm precision. Không resample trước khi chia train/test vì sẽ làm thông tin nhân tạo hoặc các điểm gần nhau lọt sang test, gây rò rỉ dữ liệu và ước lượng hiệu năng quá lạc quan.

## Kết luận chương

Ghi rõ: phương pháp imputation có RMSE thấp nhất; các detector ngoại lai đồng thuận đến đâu; scaler nào phù hợp với đặc tính phân phối; cách encoding nào tạo số chiều hợp lý; phương pháp feature selection có F1 5-fold tốt nhất; và ảnh hưởng của resampling lên precision/recall/F1-macro/AUC-ROC. Các kết luận phải lấy từ bảng/biểu đồ vừa chạy.